# 1. Подготовка окружения

In [1]:
import os
import json
from tqdm import tqdm

import numpy as np
import pandas as pd
import json

import matplotlib.pyplot as plt

from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor

from xgboost import XGBRegressor

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.base import BaseEstimator, RegressorMixin

from src import (
    MomentumTransformer2D,
    LegendreTransformer2D,
    FourierTransformer2D,
    ZernikeTransformer2D,
    import_df
)

In [2]:
torch.cuda.is_available()

True

In [3]:
RESULTS_PATH = "results/Experiment_1_3.json"
BASELINE_PATH = "results/Baseline_1_3.json"
CACHE_DIR = "cache_features"

VAL_RATIO = 2 / 7
TOL = 1e-4
N_ITER_NO_CHANGE = 500 #100
MAX_EPOCHS = 15000 #5000

TOL_CNN = 1e-4
N_ITER_NO_CHANGE_CNN = 500
MAX_EPOCHS_CNN = 15000

# 2. Загрузка данных и нормировка X
Загружаем Y из Y_ions.csv
Загружаем X (карты ФЛ) через функцию import_df
Формируем массив X размера (N, H, W)
Проверяем согласованность индексов

In [4]:
def load_results():
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH, "r") as f:
            return json.load(f)
    return {}

def save_results(results):
    dir_path = os.path.dirname(RESULTS_PATH)

    if dir_path:  # ✅ только если путь не пустой
        os.makedirs(dir_path, exist_ok=True)

    with open(RESULTS_PATH, "w") as f:
        json.dump(results, f)

def ensure_structure(results, model_name, mom):
    if model_name not in results:
        results[model_name] = {}
    if mom not in results[model_name]:
        results[model_name][mom] = {}

In [5]:
os.makedirs(CACHE_DIR, exist_ok=True)

def get_cache_path(model_name, mom, order, fold_id):
    return os.path.join(CACHE_DIR, f"{mom}_order{order}_fold{fold_id}.npz")


def load_cached_features(mom, order, fold_id):
    path = get_cache_path("", mom, order, fold_id)
    if os.path.exists(path):
        data = np.load(path)
        return data["X_train"], data["X_test"]
    return None


def save_cached_features(mom, order, fold_id, X_train_tr, X_test_tr):
    path = get_cache_path("", mom, order, fold_id)
    np.savez(path, X_train=X_train_tr, X_test=X_test_tr)

In [6]:
# Y
annotations_file = 'CD_HM_dataset/Y_ions.csv'
y_df = pd.read_csv(annotations_file, index_col='num')

# X
X_list = []
for i in tqdm(range(1, len(y_df)+1)):
    df = import_df(str(i)+'.csv')
    df = df.fillna(0)
    X_list.append(df.values)

X = np.array(X_list)

#X = np.array(range(1, len(y_df)+1)) # Заглушка
y = y_df.values

print(X.shape, y.shape)

100%|██████████| 7813/7813 [06:38<00:00, 19.59it/s]


(7813, 27, 201) (7813, 7)


In [7]:
# df.values.shape, df.values

In [8]:
X_max = X.max()
X = X / X_max

# 3. Определение трансформеров (базисов)
Параметры:
* варьируем order
* x_bounds/y_bounds фиксированы (кроме Zernike — задаем явно)

In [9]:
def get_transformer(name, order, x_bounds=(375, 575), y_bounds=(280, 410)):
    if name == "momentum":
        return MomentumTransformer2D(order=order)
    elif name == "legendre":
        return LegendreTransformer2D(order=order)
    elif name == "fourier":
        return FourierTransformer2D(order=order)
    elif name == "zernike":
        return ZernikeTransformer2D(
            order=order,
            x_bounds=x_bounds,
            y_bounds=y_bounds
        )

# 4. Подготовка моделей
Параметры:

* единый validation split (где возможно)
* фиксированные random_state

## Реализация MLP в Pytorch для ускорения обучения

In [10]:
import copy
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# =====================================================
# Torch MLP NETWORK
# =====================================================

class TorchMLPNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.Sigmoid(),   # logistic

            nn.Linear(64, 32),
            nn.Sigmoid(),

            nn.Linear(32, 16),
            nn.Sigmoid(),

            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.net(x)


# =====================================================
# TorchMLPRegressor
# =====================================================

class TorchMLPRegressor(BaseEstimator, RegressorMixin):

    def __init__(
        self,
        max_iter=MAX_EPOCHS,
        validation_fraction=VAL_RATIO,
        n_iter_no_change=N_ITER_NO_CHANGE,
        tol=TOL,

        batch_size=256,
        lr=1e-3,

        random_state=42,
        verbose=False,
        device=None
    ):

        self.max_iter = max_iter
        self.validation_fraction = validation_fraction
        self.n_iter_no_change = n_iter_no_change
        self.tol = tol

        self.batch_size = batch_size
        self.lr = lr

        self.random_state = random_state
        self.verbose = verbose
        self.device = device

    # -------------------------------------------------

    def _get_device(self):

        if self.device is not None:
            return self.device

        return "cuda" if torch.cuda.is_available() else "cpu"

    # -------------------------------------------------

    def fit(self, X, y):

        np.random.seed(self.random_state)
        torch.manual_seed(self.random_state)

        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32).reshape(-1, 1)

        # =====================================
        # train / validation split
        # =====================================
        X_train, X_val, y_train, y_val = train_test_split(
            X,
            y,
            test_size=self.validation_fraction,
            random_state=self.random_state
        )

        self.device_ = self._get_device()

        if self.verbose:
            print(f"Using device: {self.device_}")

        # =====================================
        # tensors
        # =====================================
        train_ds = TensorDataset(
            torch.tensor(X_train),
            torch.tensor(y_train)
        )

        train_loader = DataLoader(
            train_ds,
            batch_size=self.batch_size,
            shuffle=True
        )

        X_val_t = torch.tensor(X_val).to(self.device_)
        y_val_t = torch.tensor(y_val).to(self.device_)

        # =====================================
        # model
        # =====================================
        self.model_ = TorchMLPNet(X.shape[1]).to(self.device_)

        optimizer = torch.optim.Adam(
            self.model_.parameters(),
            lr=self.lr
        )

        criterion = nn.MSELoss()

        # =====================================
        # history
        # =====================================
        self.train_loss_curve_ = []
        self.val_loss_curve_ = []

        self.val_mae_curve_ = []
        self.val_rmse_curve_ = []
        self.val_r2_curve_ = []

        best_loss = np.inf
        best_state = None
        patience = 0

        # =====================================
        # training loop
        # =====================================
        for epoch in range(1, self.max_iter + 1):

            # ---------------------------
            # TRAIN
            # ---------------------------
            self.model_.train()

            batch_losses = []

            for xb, yb in train_loader:

                xb = xb.to(self.device_)
                yb = yb.to(self.device_)

                optimizer.zero_grad()

                preds = self.model_(xb)

                loss = criterion(preds, yb)

                loss.backward()

                optimizer.step()

                batch_losses.append(loss.item())

            train_loss = float(np.mean(batch_losses))

            # ---------------------------
            # VALIDATION
            # ---------------------------
            self.model_.eval()

            with torch.no_grad():

                val_preds = self.model_(X_val_t)

                val_loss = criterion(
                    val_preds,
                    y_val_t
                ).item()

                y_true = y_val_t.cpu().numpy().ravel()
                y_pred = val_preds.cpu().numpy().ravel()

                val_mae = mean_absolute_error(y_true, y_pred)
                val_rmse = np.sqrt(mean_squared_error(y_true, y_pred))
                val_r2 = r2_score(y_true, y_pred)

            # ---------------------------
            # save history
            # ---------------------------
            self.train_loss_curve_.append(train_loss)
            self.val_loss_curve_.append(val_loss)

            self.val_mae_curve_.append(val_mae)
            self.val_rmse_curve_.append(val_rmse)
            self.val_r2_curve_.append(val_r2)

            # ---------------------------
            # verbose
            # ---------------------------
            if self.verbose:
                print(
                    f"Epoch {epoch:4d}/{self.max_iter} | "
                    f"train_loss={train_loss:.6f} | "
                    f"val_loss={val_loss:.6f} | "
                    f"val_mae={val_mae:.6f} | "
                    f"val_rmse={val_rmse:.6f} | "
                    f"val_r2={val_r2:.6f}"
                )

            # ---------------------------
            # early stopping
            # ---------------------------
            if best_loss - val_loss > self.tol:

                best_loss = val_loss
                best_state = copy.deepcopy(
                    self.model_.state_dict()
                )

                patience = 0

            else:
                patience += 1

            if patience >= self.n_iter_no_change:

                if self.verbose:
                    print(
                        f"Early stopping at epoch {epoch}. "
                        f"Best val_loss={best_loss:.6f}"
                    )

                break

        # restore best model
        if best_state is not None:
            self.model_.load_state_dict(best_state)

        self.n_iter_ = epoch
        self.best_val_loss_ = best_loss

        return self

    # -------------------------------------------------

    def predict(self, X):

        X = np.asarray(X, dtype=np.float32)

        X_t = torch.tensor(X).to(self.device_)

        self.model_.eval()

        with torch.no_grad():

            preds = self.model_(X_t)

        return preds.cpu().numpy().ravel()

    # -------------------------------------------------
    # PLOT HISTORY
    # -------------------------------------------------

    def plot_history(self):

        epochs = np.arange(
            1,
            len(self.train_loss_curve_) + 1
        )

        fig, axes = plt.subplots(
            2,
            2,
            figsize=(14, 10)
        )

        # LOSS
        axes[0, 0].plot(
            epochs,
            self.train_loss_curve_,
            label="train_loss"
        )

        axes[0, 0].plot(
            epochs,
            self.val_loss_curve_,
            label="val_loss"
        )

        axes[0, 0].set_title("Loss")
        axes[0, 0].legend()
        axes[0, 0].grid(True)

        # MAE
        axes[0, 1].plot(
            epochs,
            self.val_mae_curve_
        )

        axes[0, 1].set_title("Validation MAE")
        axes[0, 1].grid(True)

        # RMSE
        axes[1, 0].plot(
            epochs,
            self.val_rmse_curve_
        )

        axes[1, 0].set_title("Validation RMSE")
        axes[1, 0].grid(True)

        # R2
        axes[1, 1].plot(
            epochs,
            self.val_r2_curve_
        )

        axes[1, 1].set_title("Validation R2")
        axes[1, 1].grid(True)

        plt.tight_layout()
        plt.show()

In [11]:
def get_model(name):
    if name == "LR":
        return LinearRegression()

    elif name == "RF":
        # ограничиваем сложность вместо early stopping
        return RandomForestRegressor(
            n_estimators=200,
            max_depth=10,
            n_jobs=-1,
            random_state=42,
        )

    elif name == "GB":
        return XGBRegressor(
            n_estimators=1000,
            learning_rate=0.05,
            max_depth=5,
            subsample=0.8,
            colsample_bytree=0.8,
            n_jobs=-1,
            random_state=42,
            early_stopping_rounds=N_ITER_NO_CHANGE,
            eval_metric="rmse"
        )

    elif name == "MLP":
        return TorchMLPRegressor(
            max_iter=MAX_EPOCHS,
            validation_fraction=VAL_RATIO,
            n_iter_no_change=N_ITER_NO_CHANGE,
            tol=TOL,
            random_state=42,
            verbose=True,
        )

## Реализация CNN-бейзлайна

Создаем класс CNNRegressor:

* наследуем BaseEstimator, RegressorMixin
* используем torch
* реализуем:
    * init
    * fit (с early stopping)
    * predict
* вход: (N, H, W) → reshape в (N, 1, H, W)

In [12]:
import copy
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# =====================================================
# CNN MODEL
# =====================================================

class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(
                in_channels=1,
                out_channels=3,
                kernel_size=5,
                stride=5
            ),

            nn.LeakyReLU(),

            nn.MaxPool2d(2),

            nn.Flatten(),

            nn.Linear(3 * 20 * 2, 32),

            nn.LeakyReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)


# =====================================================
# CNN REGRESSOR
# =====================================================

class CNNRegressor(BaseEstimator, RegressorMixin):

    def __init__(
        self,
        epochs=200,
        lr=1e-3,
        tol=1e-4,
        n_iter_no_change=10,
        batch_size=None,
        device=None,
        verbose=True
    ):

        self.epochs = epochs
        self.lr = lr
        self.tol = tol
        self.n_iter_no_change = n_iter_no_change
        self.batch_size = batch_size
        self.device = device
        self.verbose = verbose

    # -------------------------------------------------
    # DEVICE
    # -------------------------------------------------

    def _get_device(self):

        if self.device is not None:
            return self.device

        return "cuda" if torch.cuda.is_available() else "cpu"

    # -------------------------------------------------
    # FIT
    # -------------------------------------------------

    def fit(self, X, y):

        # split
        X_train, X_val, y_train, y_val = train_test_split(
            X,
            y,
            test_size=VAL_RATIO,
            random_state=42
        )

        # reshape
        X_train = torch.tensor(
            X_train[:, None, :, :],
            dtype=torch.float32
        )

        X_val = torch.tensor(
            X_val[:, None, :, :],
            dtype=torch.float32
        )

        y_train = torch.tensor(
            y_train.reshape(-1, 1),
            dtype=torch.float32
        )

        y_val = torch.tensor(
            y_val.reshape(-1, 1),
            dtype=torch.float32
        )

        # device
        self.device_ = self._get_device()

        if self.verbose:
            print(f"Using device: {self.device_}")

        X_train = X_train.to(self.device_)
        X_val = X_val.to(self.device_)

        y_train = y_train.to(self.device_)
        y_val = y_val.to(self.device_)

        # model
        self.model = CNNModel().to(self.device_)

        optimizer = optim.Adam(
            self.model.parameters(),
            lr=self.lr
        )

        loss_fn = nn.MSELoss()

        # history
        self.train_loss_curve_ = []
        self.val_loss_curve_ = []

        self.val_mae_curve_ = []
        self.val_rmse_curve_ = []
        self.val_r2_curve_ = []

        best_loss = np.inf
        best_state = None
        patience = 0

        # training
        for epoch in range(1, self.epochs + 1):

            # ==========================
            # TRAIN
            # ==========================
            self.model.train()

            optimizer.zero_grad()

            preds = self.model(X_train)

            train_loss = loss_fn(preds, y_train)

            train_loss.backward()

            optimizer.step()

            train_loss_value = train_loss.item()

            # ==========================
            # VALIDATION
            # ==========================
            self.model.eval()

            with torch.no_grad():

                val_preds = self.model(X_val)

                val_loss = loss_fn(
                    val_preds,
                    y_val
                ).item()

                y_true = y_val.cpu().numpy().ravel()
                y_pred = val_preds.cpu().numpy().ravel()

                val_mae = mean_absolute_error(y_true, y_pred)
                val_rmse = np.sqrt(mean_squared_error(y_true, y_pred))
                val_r2 = r2_score(y_true, y_pred)

            # ==========================
            # SAVE HISTORY
            # ==========================
            self.train_loss_curve_.append(train_loss_value)
            self.val_loss_curve_.append(val_loss)

            self.val_mae_curve_.append(val_mae)
            self.val_rmse_curve_.append(val_rmse)
            self.val_r2_curve_.append(val_r2)

            # ==========================
            # VERBOSE
            # ==========================
            if self.verbose:
                print(
                    f"Epoch {epoch:4d}/{self.epochs} | "
                    f"train_loss={train_loss_value:.6f} | "
                    f"val_loss={val_loss:.6f} | "
                    f"val_mae={val_mae:.6f} | "
                    f"val_rmse={val_rmse:.6f} | "
                    f"val_r2={val_r2:.6f}"
                )

            # ==========================
            # EARLY STOPPING
            # ==========================
            if best_loss - val_loss > self.tol:

                best_loss = val_loss
                patience = 0

                best_state = copy.deepcopy(
                    self.model.state_dict()
                )

            else:
                patience += 1

            if patience >= self.n_iter_no_change:

                if self.verbose:
                    print(
                        f"Early stopping at epoch {epoch}"
                    )

                break

        # restore best model
        if best_state is not None:
            self.model.load_state_dict(best_state)

        self.n_epochs_ = epoch
        self.best_val_loss_ = best_loss

        return self

    # -------------------------------------------------
    # PREDICT
    # -------------------------------------------------

    def predict(self, X):

        X = torch.tensor(
            X[:, None, :, :],
            dtype=torch.float32
        ).to(self.device_)

        self.model.eval()

        with torch.no_grad():

            preds = self.model(X)

        return preds.cpu().numpy().ravel()

    # -------------------------------------------------
    # PLOT HISTORY
    # -------------------------------------------------

    def plot_history(self):

        epochs = np.arange(
            1,
            len(self.train_loss_curve_) + 1
        )

        fig, axes = plt.subplots(
            2,
            2,
            figsize=(14, 10)
        )

        # LOSS
        axes[0, 0].plot(
            epochs,
            self.train_loss_curve_,
            label="train_loss"
        )

        axes[0, 0].plot(
            epochs,
            self.val_loss_curve_,
            label="val_loss"
        )

        axes[0, 0].set_title("Loss")
        axes[0, 0].legend()
        axes[0, 0].grid(True)

        # MAE
        axes[0, 1].plot(
            epochs,
            self.val_mae_curve_
        )

        axes[0, 1].set_title("Validation MAE")
        axes[0, 1].grid(True)

        # RMSE
        axes[1, 0].plot(
            epochs,
            self.val_rmse_curve_
        )

        axes[1, 0].set_title("Validation RMSE")
        axes[1, 0].grid(True)

        # R2
        axes[1, 1].plot(
            epochs,
            self.val_r2_curve_
        )

        axes[1, 1].set_title("Validation R2")
        axes[1, 1].grid(True)

        plt.tight_layout()
        plt.show()

# 5. Реализация пайплайна
Для каждой комбинации:

* модель
* базис
* порядок (order)

Реализовано вычисление моментов для всей БД X и их сохранение для экономии вычислительного ресурса.

In [13]:
def compute_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
    rmse = np.sqrt(mean_squared_error(y_true, y_pred, multioutput='raw_values'))
    r2 = r2_score(y_true, y_pred, multioutput='raw_values')
    return mae, rmse, r2

In [ ]:
from sklearn.model_selection import train_test_split
from xgboost import callback

kf = KFold(n_splits=10, shuffle=True, random_state=42)

models = ["LR", "RF", "GB", "MLP"]
momenta = ["momentum", "legendre", "fourier", "zernike"]
ions = ["Cu", "Ni", "Pb", "Al", "Co", "Cr", "NO3"]

results = load_results()

TOTAL_FOLDS = kf.get_n_splits()

for model_name in models:

    print("=" * 80)
    print(f"START MODEL: {model_name}")
    print("=" * 80)

    for mom in momenta:

        ensure_structure(results, model_name, mom)

        max_order = 13 if mom != "fourier" else 7

        print(f"\nMoment type: {mom} | max_order = {max_order}")

        for order in range(max_order + 1):

            print(f"\n--- {mom} | order {order}/{max_order} | model {model_name} ---")

            # создаём структуру по ионам
            for ion in ions:
                if ion not in results[model_name][mom]:
                    results[model_name][mom][ion] = {}

            for fold_id, (train_idx, test_idx) in enumerate(kf.split(X), start=1):

                # ===== признаки (кэш) =====
                cached = load_cached_features(mom, order, fold_id - 1)

                if cached is not None:
                    X_train_tr, X_test_tr = cached
                else:
                    transformer = get_transformer(mom, order)

                    X_train = X[train_idx]
                    X_test = X[test_idx]

                    transformer.fit(X_train)
                    X_train_tr = transformer.transform(X_train)
                    X_test_tr = transformer.transform(X_test)

                    save_cached_features(
                        mom,
                        order,
                        fold_id - 1,
                        X_train_tr,
                        X_test_tr
                    )

                n_features = str(X_train_tr.shape[1])

                # ===== цикл по ионам =====
                for ion_idx, ion_name in enumerate(ions):

                    # если уже полностью посчитано — пропускаем
                    if n_features in results[model_name][mom][ion_name]:
                        if len(results[model_name][mom][ion_name][n_features]["MAE"]) >= TOTAL_FOLDS:
                            continue

                    print(
                        f"RUN -> model={model_name} | "
                        f"moment={mom} | "
                        f"order={order}/{max_order} | "
                        f"ion={ion_name} ({ion_idx+1}/{len(ions)}) | "
                        f"fold={fold_id}/{TOTAL_FOLDS}"
                    )

                    # ===== данные =====
                    X_train = X[train_idx]
                    X_test = X[test_idx]

                    y_train = y[train_idx, ion_idx].reshape(-1, 1)
                    y_test = y[test_idx, ion_idx].reshape(-1, 1)

                    scaler = MinMaxScaler()
                    y_train_s = scaler.fit_transform(y_train)

                    # split train -> val
                    X_tr, X_val, y_tr, y_val = train_test_split(
                        X_train_tr,
                        y_train_s,
                        test_size=VAL_RATIO,
                        random_state=42
                    )

                    model = get_model(model_name)

                    # ===== обучение =====
                    if model_name == "GB":

                        model.fit(
                            X_tr,
                            y_tr.ravel(),
                            eval_set=[(X_val, y_val.ravel())],
                            verbose=False
                        )

                    elif model_name == "MLP":

                        model.fit(
                            X_train_tr,
                            y_train_s.ravel()
                        )
                        
                        model.plot_history()

                    else:
                        model.fit(
                            X_train_tr,
                            y_train_s.ravel()
                        )

                    # ===== predict =====
                    preds = model.predict(X_test_tr).reshape(-1, 1)
                    preds = scaler.inverse_transform(preds)

                    # ===== метрики =====
                    mae = mean_absolute_error(y_test, preds)
                    rmse = np.sqrt(mean_squared_error(y_test, preds))
                    r2 = r2_score(y_test, preds)

                    # ===== структура =====
                    if n_features not in results[model_name][mom][ion_name]:
                        results[model_name][mom][ion_name][n_features] = {
                            "MAE": [],
                            "RMSE": [],
                            "R2": []
                        }

                    results[model_name][mom][ion_name][n_features]["MAE"].append(float(mae))
                    results[model_name][mom][ion_name][n_features]["RMSE"].append(float(rmse))
                    results[model_name][mom][ion_name][n_features]["R2"].append(float(r2))

                # сохраняем после каждого фолда
                save_results(results)

print("\nALL EXPERIMENTS FINISHED")

START MODEL: LR

Moment type: momentum | max_order = 13

--- momentum | order 0/13 | model LR ---

--- momentum | order 1/13 | model LR ---

--- momentum | order 2/13 | model LR ---

--- momentum | order 3/13 | model LR ---

--- momentum | order 4/13 | model LR ---

--- momentum | order 5/13 | model LR ---

--- momentum | order 6/13 | model LR ---

--- momentum | order 7/13 | model LR ---

--- momentum | order 8/13 | model LR ---

--- momentum | order 9/13 | model LR ---

--- momentum | order 10/13 | model LR ---

--- momentum | order 11/13 | model LR ---

--- momentum | order 12/13 | model LR ---

--- momentum | order 13/13 | model LR ---

Moment type: legendre | max_order = 13

--- legendre | order 0/13 | model LR ---

--- legendre | order 1/13 | model LR ---

--- legendre | order 2/13 | model LR ---

--- legendre | order 3/13 | model LR ---

--- legendre | order 4/13 | model LR ---

--- legendre | order 5/13 | model LR ---

--- legendre | order 6/13 | model LR ---

--- legendre | or

## Бейзлайн CNN
* обучаем аналогично CV
* сохраняем метрики отдельно

In [ ]:
ions = ["Cu", "Ni", "Pb", "Al", "Co", "Cr", "NO3"]

kf = KFold(n_splits=10, shuffle=True, random_state=42)


# загрузка
if os.path.exists(BASELINE_PATH):
    with open(BASELINE_PATH, "r") as f:
        baseline = json.load(f)
else:
    baseline = {ion: {"MAE": [], "RMSE": [], "R2": []} for ion in ions}

for ion_idx, ion_name in enumerate(ions):

    for fold_id, (train_idx, test_idx) in enumerate(
        tqdm(kf.split(X), total=10, desc=f"CNN baseline | {ion_name}")
    ):

        # пропуск уже посчитанного
        if len(baseline[ion_name]["MAE"]) > fold_id:
            continue

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx, ion_idx], y[test_idx, ion_idx]

        y_train = y_train.reshape(-1, 1)
        y_test = y_test.reshape(-1, 1)

        scaler = MinMaxScaler()
        y_train_s = scaler.fit_transform(y_train)

        model = CNNRegressor(
            epochs=MAX_EPOCHS_CNN,
            tol=TOL_CNN,
            n_iter_no_change=N_ITER_NO_CHANGE_CNN
        )

        model.fit(X_train, y_train_s)
        model.plot_history()

        preds = model.predict(X_test).reshape(-1, 1)
        preds = scaler.inverse_transform(preds)

        mae = mean_absolute_error(y_test, preds)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        r2 = r2_score(y_test, preds)

        baseline[ion_name]["MAE"].append(mae)
        baseline[ion_name]["RMSE"].append(rmse)
        baseline[ion_name]["R2"].append(r2)

        # сохраняем после каждого фолда
        dir_path = os.path.dirname(BASELINE_PATH)

        if dir_path:  # ✅ только если путь не пустой
            os.makedirs(dir_path, exist_ok=True)

        with open(BASELINE_PATH, "w") as f:
            json.dump(baseline, f)

# финальная статистика
baseline_stats = {}

for ion in ions:
    baseline_stats[ion] = {
        "MAE_mean": np.mean(baseline[ion]["MAE"]),
        "MAE_std": np.std(baseline[ion]["MAE"]),
        "RMSE_mean": np.mean(baseline[ion]["RMSE"]),
        "RMSE_std": np.std(baseline[ion]["RMSE"]),
        "R2_mean": np.mean(baseline[ion]["R2"]),
        "R2_std": np.std(baseline[ion]["R2"]),
    }

# 6. Анализ результатов

Для каждой модели:

* 7 подграфиков (по ионам)
    * X: n_features
    * Y: mean MAE
    * error bars: std
    * линии по базисам
        * baseline:
        * горизонтальная линия mean
        * полупрозрачная зона std

In [ ]:
ions = ["Cu", "Ni", "Pb", "Al", "Co", "Cr", "NO3"]

# загрузка results
with open(RESULTS_PATH, "r") as f:
    results = json.load(f)

# загрузка baseline
with open(BASELINE_PATH, "r") as f:
    baseline = json.load(f)

for model_name in results:

    fig, axes = plt.subplots(7, 1, figsize=(10, 22), sharex=True)

    for i, ion in enumerate(ions):
        ax = axes[i]

        for mom in momenta:
            xs = []
            ys = []
            errs = []

            if ion not in results[model_name][mom]:
                continue

            for n_feat in sorted(results[model_name][mom][ion].keys(), key=lambda x: int(x)):

                data = np.array(results[model_name][mom][ion][n_feat]["MAE"])

                xs.append(int(n_feat))
                ys.append(np.mean(data))
                errs.append(np.std(data))

            if len(xs) > 0:
                ax.errorbar(xs, ys, yerr=errs, marker='o', label=mom)

        # baseline
        baseline_mean = np.mean(baseline[ion]["MAE"])
        baseline_std = np.std(baseline[ion]["MAE"])

        ax.axhline(baseline_mean, linestyle="--", color="gray")

        ax.fill_between(
            xs,
            baseline_mean - baseline_std,
            baseline_mean + baseline_std,
            color="gray",
            alpha=0.2
        )

        ax.set_title(f"{model_name} - {ion}")
        ax.set_ylabel("MAE")
        ax.legend()

    axes[-1].set_xlabel("n_features")
    #axes[0].legend()

    plt.tight_layout()
    plt.show()